In [1]:
#pip install rdkit

In [2]:
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client
import rdkit
from rdkit import Chem 

In [6]:
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, Crippen, Lipinski


ligand_ids = [
    "CHEMBL1079742",  # erlotinib
    "CHEMBL941",      # gefitinib
    "CHEMBL2105758"   # osimertinib
]
molecule = new_client.molecule 

mol_res = molecule.filter(molecule_chembl_id__in=ligand_ids)
mol_df = pd.DataFrame(mol_res)

def get_smiles(structures):
    if isinstance(structures, dict):
        return structures.get("canonical_smiles")
    return None

mol_df["canonical_smiles"] = mol_df["molecule_structures"].apply(get_smiles)

def compute_smiles_features(smiles):
    if pd.isna(smiles):
        return {
            "canonical_smiles": None,
            "MolWt": np.nan,
            "MolLogP": np.nan,
            "TPSA": np.nan,
            "NumHDonors": np.nan,
            "NumHAcceptors": np.nan,
            "NumRotatableBonds": np.nan,
            "RingCount": np.nan,
            "HeavyAtomCount": np.nan
        }

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {
            "canonical_smiles": smiles,
            "MolWt": np.nan,
            "MolLogP": np.nan,
            "TPSA": np.nan,
            "NumHDonors": np.nan,
            "NumHAcceptors": np.nan,
            "NumRotatableBonds": np.nan,
            "RingCount": np.nan,
            "HeavyAtomCount": np.nan
        }

    return {
        "canonical_smiles": Chem.CanonSmiles(smiles),
        "MolWt": Descriptors.MolWt(mol),
        "MolLogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "NumHDonors": Lipinski.NumHDonors(mol),
        "NumHAcceptors": Lipinski.NumHAcceptors(mol),
        "NumRotatableBonds": Lipinski.NumRotatableBonds(mol),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "HeavyAtomCount": mol.GetNumHeavyAtoms()
    }

smiles_features = pd.DataFrame(
    [compute_smiles_features(s) for s in mol_df["canonical_smiles"]]
)

ligand_features = pd.concat(
    [mol_df[["molecule_chembl_id", "pref_name"]].rename(columns={"molecule_chembl_id": "ligand"}), smiles_features],
    axis=1
).drop_duplicates(subset=["ligand"])

print(ligand_features.head())
ligand_SMILES = ligand_features.to_csv("data/ligand_smiles_features.csv", index=False)

          ligand                pref_name  \
0      CHEMBL941                 IMATINIB   
1  CHEMBL1079742  ERLOTINIB HYDROCHLORIDE   
2  CHEMBL2105758     AVATROMBOPAG MALEATE   

                                    canonical_smiles    MolWt  MolLogP  \
0  Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc...  493.615  4.59032   
1      C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1.Cl  429.904  3.82690   
2  O=C(Nc1nc(-c2cc(Cl)cs2)c(N2CCN(C3CCCCC3)CC2)s1...  765.742  6.29320   

     TPSA  NumHDonors  NumHAcceptors  NumRotatableBonds  RingCount  \
0   86.28           2              7                  7          5   
1   74.73           1              7                 10          3   
2  176.50           4             11                  9          6   

   HeavyAtomCount  
0              37  
1              30  
2              50  
